# Recurrent LIF Network — inhibitory strength comparison

Runs the original recurrent leaky integrate-and-fire network for several values of `g`, the relative inhibitory synaptic strength. Excitatory events add `J`; inhibitory events add `-g*J`, matching the implementation and the 2014 reference.

The full network is opt-in because it contains 125,000 neurons. Set `RUN_SWEEP = True` only for a deliberate run.


In [5]:
! pip install brian2 -q

In [ ]:
# Keep numerical outputs and figures inside 01_Original_Colab/results.
from pathlib import Path

ROOT_CANDIDATES = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / "01_Original_Colab").is_dir()),
    next((p.parent for p in ROOT_CANDIDATES if p.name == "01_Original_Colab"), Path.cwd()),
)
RESULTS_ROOT = ROOT / "01_Original_Colab" / "results"
CONDITION = "comparison"
OUTPUT_DIR = RESULTS_ROOT / CONDITION
FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [6]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np

try:
    from brian2 import (NeuronGroup, PopulationRateMonitor, SpikeMonitor,
                        Synapses, defaultclock, mV, ms, prefs, run, seed,
                        second, start_scope)
except ImportError as exc:
    raise ImportError("Install requirements.txt before running this notebook.") from exc

RUN_SWEEP = False
USE_CYTHON = True
SAVE_RESULTS = True
SAVE_FIGURE = True
G_VALUES = [3.0, 4.0, 4.5, 6.0]
SEED = 4321
SIM_TIME_MS = 3000.0

# Original recurrent LIF network parameters.
N_E, N_I = 100000, 25000
C_E, C_I = 1000, 250
TAU_MS, THETA_MV, V_RESET_MV = 20.0, 20.0, 10.0
REFRACTORY_MS, J_MV, DELAY_MS = 2.0, 0.1, 1.5
EXTERNAL_INPUT_MV, DT_MS = 30.0, 0.1

ROOT_CANDIDATES = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / "01_Original_Colab").is_dir()),
    next((p.parent for p in ROOT_CANDIDATES if p.name == "01_Original_Colab"), Path.cwd()),
)
OUTPUT_DIR = ROOT / "01_Original_Colab" / "results" / "comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if USE_CYTHON:
    prefs.codegen.target = "cython"


## Parameterized network run

The same seed is used for each condition, so changing `g` is the primary difference between runs. The optional small-network arguments make it possible to validate the notebook without allocating the full model.

In [7]:
def run_condition(g_value, *, n_e=N_E, n_i=N_I, c_e=C_E, c_i=C_I,
                  duration_ms=SIM_TIME_MS, seed_value=SEED):
    """Run one condition and return mean firing rate and mean CV."""
    if g_value < 0 or min(n_e, n_i, c_e, c_i) <= 0:
        raise ValueError("g and network counts must be positive")
    if c_e > n_e or c_i > n_i:
        raise ValueError("Incoming counts cannot exceed source populations")
    start_scope()
    defaultclock.dt = DT_MS * ms
    seed(seed_value)
    np.random.seed(seed_value)
    n_total = n_e + n_i
    tau, theta = TAU_MS * ms, THETA_MV * mV
    v_reset, refractory = V_RESET_MV * mV, REFRACTORY_MS * ms
    psp, delay = J_MV * mV, DELAY_MS * ms
    external_input = EXTERNAL_INPUT_MV * mV
    neurons = NeuronGroup(
        n_total,
        model="dv/dt = (-v + external_input) / tau : volt (unless refractory)",
        threshold="v > theta", reset="v = v_reset", refractory=refractory,
        method="euler")
    neurons.v = "rand() * mV"
    excitatory, inhibitory = neurons[:n_e], neurons[n_e:]
    exc_synapses = Synapses(excitatory, neurons, on_pre="v += psp", delay=delay)
    inh_synapses = Synapses(inhibitory, neurons,
                            on_pre="v -= inhibition_factor * psp", delay=delay,
                            namespace={"inhibition_factor": float(g_value), "psp": psp})
    targets_e, targets_i = np.repeat(np.arange(n_total), c_e), np.repeat(np.arange(n_total), c_i)
    exc_synapses.connect(i=np.random.randint(0, n_e, n_total * c_e), j=targets_e)
    inh_synapses.connect(i=np.random.randint(0, n_i, n_total * c_i), j=targets_i)
    rate_monitor, spike_monitor = PopulationRateMonitor(neurons), SpikeMonitor(neurons)
    run(duration_ms * ms)
    mean_rate_hz = float(np.mean(np.asarray(rate_monitor.rate / (1 * second))))
    spike_trains = spike_monitor.spike_trains()
    cv_values = []
    for neuron_index, train in spike_trains.items():
        intervals = np.diff(np.asarray(train / ms))
        if intervals.size and np.mean(intervals) > 0:
            cv_values.append(float(np.std(intervals) / np.mean(intervals)))
    mean_cv = float(np.mean(cv_values)) if cv_values else float("nan")
    return {"g": float(g_value), "mean_rate_hz": mean_rate_hz, "mean_cv": mean_cv}


In [8]:
# Run a small explicit check before a full sweep, if desired.
SMALL_TEST = False
if SMALL_TEST:
    print(run_condition(4.0, n_e=20, n_i=5, c_e=4, c_i=2,
                        duration_ms=20.0))

if RUN_SWEEP:
    results = [run_condition(g_value) for g_value in G_VALUES]
    for item in results:
        print(f"g={item['g']:.1f}: rate={item['mean_rate_hz']:.2f} Hz, CV={item['mean_cv']:.3f}")
    if SAVE_RESULTS:
        (OUTPUT_DIR / "Recurrent_LIF_G_COMPARISON_results.json").write_text(json.dumps({
            "g_values": G_VALUES, "seed": SEED, "duration_ms": SIM_TIME_MS,
            "N_E": N_E, "N_I": N_I, "C_E": C_E, "C_I": C_I,
            "results": results}, indent=2) + "\n")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot([x["g"] for x in results], [x["mean_rate_hz"] for x in results], "o-")
    axes[1].plot([x["g"] for x in results], [x["mean_cv"] for x in results], "o-")
    axes[0].set(xlabel="g (relative inhibition)", ylabel="Mean firing rate (Hz)")
    axes[1].set(xlabel="g (relative inhibition)", ylabel="Mean CV")
    fig.tight_layout()
    if SAVE_FIGURE:
        fig.savefig(OUTPUT_DIR / "Recurrent_LIF_G_COMPARISON.png", dpi=150)
    fig.savefig(FIGURES_DIR / "comparison_cell_6.png", dpi=150)
    plt.show()
else:
    print("Sweep disabled. Set RUN_SWEEP = True to run all g values.")


Sweep disabled. Set RUN_SWEEP = True to run all g values.
